# RAG — NLP TD4

A rendre :
- Le notebook
- `chunks_embeddings.csv` (chunk, embedding)
- `questions_rag.csv` (question, embedding, rag_reply)

In [30]:
import json
import os
import time
import numpy as np
import pandas as pd
from pathlib import Path

In [31]:
# ── Load raw documents ──────────────────────────────────────────────────────
path = Path("../notebooks/rag_data/rag")
if not path.exists():
    path = Path("rag_data/rag")
if not path.exists():
    path = Path("rag_data")

texts = []
for filename in sorted(path.glob("*.md")):
    with open(filename, encoding="utf-8") as f:
        texts.append(f.read())
print(f"Loaded {len(texts)} documents from {path.resolve()}")

Loaded 7 documents from C:\Users\Public\NLP-TD3\nlp_esgi\notebooks\rag_data\rag


In [32]:
# ── Chunking strategies ──────────────────────────────────────────────────────

def chunk_by_paragraph(text):
    """Split on blank lines."""
    return [c.strip() for c in text.split("\n\n") if c.strip()]

def chunk_with_title(text):
    """Same as paragraph but prepend the document title to each chunk."""
    parts = text.split("\n\n")
    title = parts[0].replace("# Title: ", "").replace("# ", "").strip()
    return [f"{title}: {c.strip()}" for c in parts if c.strip()]

def chunk_sliding_window(text, window=200, overlap=50):
    """Sliding window over tokens (words)."""
    words = text.split()
    step  = window - overlap
    chunks = []
    for i in range(0, max(1, len(words) - overlap), step):
        chunk = " ".join(words[i: i + window])
        if chunk.strip():
            chunks.append(chunk)
    return chunks

def chunk_by_section(text):
    """Split on bold section headers (**…**)."""
    lines    = text.splitlines()
    sections, current = [], []
    for line in lines:
        if line.startswith("**") and current:
            sections.append("\n".join(current).strip())
            current = [line]
        else:
            current.append(line)
    if current:
        sections.append("\n".join(current).strip())
    return [s for s in sections if s]

chunks_paragraph  = sum((chunk_by_paragraph(t)     for t in texts), [])
chunks_with_title = sum((chunk_with_title(t)        for t in texts), [])
chunks_sliding    = sum((chunk_sliding_window(t)    for t in texts), [])
chunks_section    = sum((chunk_by_section(t)        for t in texts), [])

print(f"by_paragraph:   {len(chunks_paragraph)} chunks")
print(f"with_title:     {len(chunks_with_title)} chunks")
print(f"sliding_window: {len(chunks_sliding)} chunks")
print(f"by_section:     {len(chunks_section)} chunks")

by_paragraph:   98 chunks
with_title:     98 chunks
sliding_window: 14 chunks
by_section:     49 chunks


In [33]:
pip install -q transformers==4.56.0 flagembedding

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
from FlagEmbedding import FlagModel

model = FlagModel(
    "BAAI/bge-base-en-v1.5",
    query_instruction_for_retrieval="Represent this sentence for searching relevant passages:",
    use_fp16=True,
)

In [35]:
# ── Embed all chunk corpora ──────────────────────────────────────────────────
emb_paragraph  = model.encode(chunks_paragraph)
emb_with_title = model.encode(chunks_with_title)
emb_sliding    = model.encode(chunks_sliding)
emb_section    = model.encode(chunks_section)

print(f"paragraph={emb_paragraph.shape}  with_title={emb_with_title.shape}  "
      f"sliding={emb_sliding.shape}  section={emb_section.shape}")

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


paragraph=(98, 768)  with_title=(98, 768)  sliding=(14, 768)  section=(49, 768)


In [36]:
# ── Load evaluation set (question + expected keyword in chunk) ───────────────
csv_path = path.parent / "question_answer_short.csv"
if not csv_path.exists():
    csv_path = path / "question_answer_short.csv"
df_eval = pd.read_csv(csv_path, quoting=1)
print(f"Loaded {len(df_eval)} evaluation questions")
df_eval.head()

Loaded 15 evaluation questions


,question,answer
0,Who is the reinforcement learning teacher?,Dr. Arjun Patel
1,Who teaches NLP?,Dr. Evelyn Chang
2,What tools are used in the DevOps course?,"Docker, Kubernetes, Jenkins, and Terraform"
3,When are the NLP lab sessions?,Fridays 2:00 PM - 4:00 PM
4,What RL algorithms are covered?,"Q-Learning, Deep Q-Networks (DQN)"


In [37]:
# ── MRR evaluation for basic strategies ─────────────────────────────────────

def compute_mrr(sim_scores, acceptable_chunks):
    """Mean Reciprocal Rank (cutoff @5)."""
    ranks = []
    for score, ok in zip(sim_scores, acceptable_chunks):
        indexes = list(reversed(np.argsort(score)))
        rank    = 1 + next(i for i, idx in enumerate(indexes) if idx in ok)
        ranks.append(rank)
    mrr = sum(1 / r if r <= 5 else 0 for r in ranks) / len(ranks)
    return {"score": mrr, "ranks": ranks}

strategies = {
    "by_paragraph":   (chunks_paragraph,  emb_paragraph),
    "with_title":     (chunks_with_title, emb_with_title),
    "sliding_window": (chunks_sliding,    emb_sliding),
    "by_section":     (chunks_section,    emb_section),
}

q_emb_eval  = model.encode(list(df_eval["question"]))
results_mrr = {}

for name, (corpus, corp_emb) in strategies.items():
    acceptable = [
        set(i for i, c in enumerate(corpus) if ans.lower() in c.lower())
        for ans in df_eval["answer"]
    ]
    sim = q_emb_eval @ corp_emb.T
    res = compute_mrr(sim, acceptable)
    results_mrr[name] = res["score"]
    print(f"  {name:<20}  MRR = {res['score']:.3f}")

best_strategy         = max(results_mrr, key=results_mrr.get)
best_chunks, best_emb = strategies[best_strategy]
print(f"\nBest strategy so far: {best_strategy}  MRR={results_mrr[best_strategy]:.3f}")

  by_paragraph          MRR = 0.397
  with_title            MRR = 0.833
  sliding_window        MRR = 0.967
  by_section            MRR = 0.518

Best strategy so far: sliding_window  MRR=0.967


In [38]:
# ── Small-to-Big strategy ────────────────────────────────────────────────────

def build_small2big(text, small_window=50, small_overlap=10):
    parts  = text.split("\n\n")
    title  = parts[0].replace("# Title: ", "").replace("# ", "").strip()
    large_chunks          = [f"{title}: {p.strip()}" for p in parts if p.strip()]
    small_chunks, parent_indices = [], []
    for parent_idx, large in enumerate(large_chunks):
        words = large.split()
        step  = small_window - small_overlap
        for i in range(0, max(1, len(words) - small_overlap), step):
            small = " ".join(words[i: i + small_window])
            if small.strip():
                small_chunks.append(small)
                parent_indices.append(parent_idx)
    return large_chunks, small_chunks, parent_indices

all_large, all_small, all_parents = [], [], []
offset = 0
for text in texts:
    large, small, parents = build_small2big(text)
    all_large.extend(large)
    all_small.extend(small)
    all_parents.extend([p + offset for p in parents])
    offset += len(large)

print(f"Small2Big: {len(all_small)} small chunks → {len(all_large)} large chunks")

emb_small = model.encode(all_small)

acceptable_s2b = [
    set(i for i, c in enumerate(all_large) if ans.lower() in c.lower())
    for ans in df_eval["answer"]
]
sim_s2b = q_emb_eval @ emb_small.T

def compute_mrr_s2b(sim_scores, parent_indices, acceptable_large):
    ranks = []
    for score, ok_large in zip(sim_scores, acceptable_large):
        sorted_small = list(reversed(np.argsort(score)))
        seen_large   = []
        for si in sorted_small:
            li = parent_indices[si]
            if li not in seen_large:
                seen_large.append(li)
        rank = 1 + next(
            (i for i, li in enumerate(seen_large) if li in ok_large),
            len(seen_large),
        )
        ranks.append(rank)
    return {"score": sum(1 / r if r <= 5 else 0 for r in ranks) / len(ranks), "ranks": ranks}

res_s2b                  = compute_mrr_s2b(sim_s2b, all_parents, acceptable_s2b)
results_mrr["small2big"] = res_s2b["score"]

print("\nMRR Summary (all strategies):")
for name, score in sorted(results_mrr.items(), key=lambda x: -x[1]):
    marker = " ◀ best" if score == max(results_mrr.values()) else ""
    print(f"  {name:<20}  {score:.3f}{marker}")

# Choose best overall
best_strategy = max(results_mrr, key=results_mrr.get)
if best_strategy == "small2big":
    best_chunks = all_large
    best_emb    = model.encode(all_large)
else:
    best_chunks, best_emb = strategies[best_strategy]
print(f"\n✔  Best strategy: {best_strategy}  MRR={results_mrr[best_strategy]:.3f}")

Small2Big: 105 small chunks → 98 large chunks

MRR Summary (all strategies):
  sliding_window        0.967 ◀ best
  with_title            0.833
  small2big             0.783
  by_section            0.518
  by_paragraph          0.397

✔  Best strategy: sliding_window  MRR=0.967


In [39]:
# ── Groq client ─────────────────────────────────────────────────────────────
import openai

groq_api_key = os.getenv("GROQ_API_KEY", "")
# llama-3.1-8b-instant : 20 000 TPM / 30 RPM → limites bien plus hautes que 70b
GROQ_MODEL   = "llama-3.1-8b-instant"
client = openai.OpenAI(
    api_key  = groq_api_key,
    base_url = "https://api.groq.com/openai/v1",
)

In [40]:
# ── RAG helpers ──────────────────────────────────────────────────────────────

def get_context(query, corpus, corp_emb, top_k=5):
    q_vec   = model.encode([query])
    sims    = (q_vec @ corp_emb.T)[0]
    top_idx = np.argsort(sims)[-top_k:][::-1]
    return [corpus[i] for i in top_idx]

def build_rag_prompt(query, context_chunks):
    context_str = "\n\n---\n\n".join(context_chunks)
    return (
        "You are a helpful assistant for a Computer Science university.\n"
        "Answer the student's question using ONLY the context below.\n"
        "If the answer is not in the context, reply: 'I cannot answer that question.'\n\n"
        f"Context:\n---------------------\n{context_str}\n---------------------\n\n"
        f"Question: {query}\nAnswer:"
    )

def rag_reply(query, corpus, corp_emb, top_k=3, max_retries=8):
    """Call Groq with exponential backoff on RateLimitError."""
    context_chunks = get_context(query, corpus, corp_emb, top_k)
    prompt         = build_rag_prompt(query, context_chunks)
    for attempt in range(max_retries):
        try:
            res = client.chat.completions.create(
                messages    = [{"role": "user", "content": prompt}],
                model       = GROQ_MODEL,
                temperature = 0,
                max_tokens  = 200,
            )
            return res.choices[0].message.content.strip(), context_chunks
        except Exception as e:
            if "rate_limit" in str(e).lower() or "429" in str(e):
                wait = 15 * (2 ** attempt)   # 15, 30, 60, 120, 240 … s
                print(f"  [rate-limit] waiting {wait}s (retry {attempt+1}/{max_retries})…")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f"Groq rate-limit persists after {max_retries} retries")

# Smoke test
ans, _ = rag_reply("Who is the NLP teacher?", best_chunks, best_emb)
print("Q: Who is the NLP teacher?")
print("A:", ans)

Q: Who is the NLP teacher?
A: Dr. Evelyn Chang.


In [41]:
# ── Load professor questions from question (2).csv ───────────────────────────

def load_questions_csv(filepath):
    for sep in [",", ";"]:
        try:
            tmp = pd.read_csv(filepath, sep=sep)
            col = "question" if "question" in tmp.columns else tmp.columns[0]
            return [str(q).strip() for q in tmp[col]
                    if str(q).strip() and str(q).strip().lower() != "nan"]
        except Exception:
            pass
    with open(filepath, encoding="utf-8") as f:
        lines = [l.strip() for l in f if l.strip()]
    if lines and lines[0].lower() in ("question", "questions"):
        lines = lines[1:]
    return lines

# Try several candidate locations (notebook can be run from different cwds)
candidates = [
    path.parent.parent.parent / "question (2).csv",   # nlp_esgi/
    path.parent.parent        / "question (2).csv",   # notebooks/
    path.parent               / "question (2).csv",   # rag_data/
    Path("../question (2).csv"),
    Path("question (2).csv"),
]
prof_questions_path = next((p for p in candidates if p.exists()), None)

if prof_questions_path:
    final_questions = load_questions_csv(prof_questions_path)
    print(f"✔ Loaded {len(final_questions)} questions from:\n  {prof_questions_path.resolve()}")
else:
    final_questions = list(df_eval["question"])
    print(f"⚠ question (2).csv not found — fallback to {len(final_questions)} eval questions")

print()
for i, q in enumerate(final_questions, 1):
    print(f"  [{i:02d}] {q}")

✔ Loaded 27 questions from:
  C:\Users\Public\NLP-TD3\nlp_esgi\question (2).csv

  [01] Who is the reinforcement learning teacher?
  [02] In what class will I learn game A.I.?
  [03] What are the requirements to build a game A.I.?
  [04] How will I validate the reinforcement learning class?
  [05] Which class will teach me to build a chatbot?
  [06] What are the requirements to build a chatbot?
  [07] What models do we use for text A.I.?
  [08] What are the applications of NLP?
  [09] What class will teach me to program a Arduino?
  [10] What IoT system can I build in class
  [11] What are the applications of IoT?
  [12] How do I validate the IoT class?
  [13] What language must I know to code embedded systems
  [14] Which class will have lessons on hardware
  [15] What are the specific issues I see in embedded systems?
  [16] What can I build in the embedded system class?
  [17] What class will teach me deployment procedures?
  [18] What technology are taught to deploy solutions?
  [1

In [42]:
# ── Run RAG on all 27 questions ──────────────────────────────────────────────
rag_replies  = []
q_embeddings = []

for i, q in enumerate(final_questions, 1):
    reply, _ = rag_reply(q, best_chunks, best_emb)
    q_vec    = model.encode([q])[0]
    rag_replies.append(reply)
    q_embeddings.append(q_vec)
    print(f"[{i:02d}/{len(final_questions)}] Q: {q}")
    print(f"           A: {reply[:120]}{'...' if len(reply) > 120 else ''}\n")
    time.sleep(5)   # 5 s gap → ~12 req/min, safe under Groq free-tier

print("Done.")

[01/27] Q: Who is the reinforcement learning teacher?
           A: Dr. Arjun Patel.

[02/27] Q: In what class will I learn game A.I.?
           A: You will learn game A.I. in the "Foundations of Reinforcement Learning" course, specifically in the section "Application...

[03/27] Q: What are the requirements to build a game A.I.?
           A: I cannot answer that question.

[04/27] Q: How will I validate the reinforcement learning class?
           A: You will validate the reinforcement learning class through the following assessments:

- Weekly quizzes and coding exerc...

[05/27] Q: Which class will teach me to build a chatbot?
           A: You will learn to build a chatbot in the "Introduction to NLP" course, specifically in the "Applications of NLP" section...

[06/27] Q: What are the requirements to build a chatbot?
           A: I cannot answer that question.

[07/27] Q: What models do we use for text A.I.?
           A: In the context of the Natural Language Processing (NLP) 

In [43]:
# ── Save deliverables ────────────────────────────────────────────────────────
output_dir = path.parent          # rag_data/
output_dir.mkdir(parents=True, exist_ok=True)

# 1) chunks_embeddings.csv
df_chunks = pd.DataFrame({
    "chunk":     best_chunks,
    "embedding": [json.dumps(e.tolist()) for e in best_emb],
})
df_chunks.to_csv(output_dir / "chunks_embeddings.csv", index=False)
print(f"chunks_embeddings.csv  → {len(df_chunks)} rows saved")

# 2) questions_rag.csv
df_questions = pd.DataFrame({
    "question":  final_questions,
    "embedding": [json.dumps(e.tolist()) for e in q_embeddings],
    "rag_reply": rag_replies,
})
df_questions.to_csv(output_dir / "questions_rag.csv", index=False)
print(f"questions_rag.csv      → {len(df_questions)} rows saved")

# Sanity checks
assert isinstance(json.loads(df_chunks.iloc[0]["embedding"]), list),    "chunk embedding is not a list"
assert isinstance(json.loads(df_questions.iloc[0]["embedding"]), list), "question embedding is not a list"
print("Embeddings verified: json.loads(embedding) → list[float]  ✔\n")

# Pretty preview
print("── questions_rag.csv preview ──────────────────────────────────────────")
for _, row in df_questions.iterrows():
    print(f"Q: {row['question']}")
    print(f"A: {row['rag_reply'][:200]}{'...' if len(row['rag_reply']) > 200 else ''}")
    print()

chunks_embeddings.csv  → 14 rows saved
questions_rag.csv      → 27 rows saved
Embeddings verified: json.loads(embedding) → list[float]  ✔

── questions_rag.csv preview ──────────────────────────────────────────
Q: Who is the reinforcement learning teacher?
A: Dr. Arjun Patel.

Q: In what class will I learn game A.I.?
A: You will learn game A.I. in the "Foundations of Reinforcement Learning" course, specifically in the section "Applications of RL" where it mentions "Game-playing agents (e.g., OpenAI Gym environments)"...

Q: What are the requirements to build a game A.I.?
A: I cannot answer that question.

Q: How will I validate the reinforcement learning class?
A: You will validate the reinforcement learning class through the following assessments:

- Weekly quizzes and coding exercises (20%)
- Midterm exam: Theoretical RL concepts (20%)
- Final project: Design...

Q: Which class will teach me to build a chatbot?
A: You will learn to build a chatbot in the "Introduction to NLP" course,